This is expected to be run after runnning `scripts/data_preprocess.py` and optionally after running `scripts/augment_train_dataset.py`

The goal is to explore the dataset after its been preprocessed.

In [ ]:
import os
import torch
from matplotlib import pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
train_data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "train")
valid_data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "val")
test_data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "test")

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resize all images to 224x224. To check for Resnet34 model compatibility
    transforms.ToTensor(),
])

# Load the dataset with basic transform
train_dataset = datasets.ImageFolder(train_data_dir, transform=basic_transform)
valid_dataset = datasets.ImageFolder(valid_data_dir, transform=basic_transform)
test_dataset = datasets.ImageFolder(test_data_dir, transform=basic_transform)

In [ ]:
num_classes = len(train_dataset.classes)
print('Number of classes in train dataset:', num_classes)

num_classes = len(valid_dataset.classes)
print('Number of classes in valid dataset:', num_classes)

num_classes = len(test_dataset.classes)
print('Number of classes in test dataset:', num_classes)

In [ ]:
print('Class names in train dataset:', train_dataset.classes)
print('Class names in valid dataset:', valid_dataset.classes)
print('Class names in test dataset:', test_dataset.classes)

In [ ]:
# this is the total number of training samples across all classes
len(train_dataset.targets)

In [ ]:
fig = plt.figure(figsize=(12, 14))
for i in range(num_classes):
    ax = fig.add_subplot(7, 5, i + 1, xticks=[], yticks=[])
    idx = train_dataset.targets.index(i)
    img = train_dataset[idx][0]
    ax.set_title(train_dataset.classes[i])
    # Convert from CxHxW to HxWxC for display
    plt.imshow(img.permute(1, 2, 0)) 
plt.show()

224x224 resizing looks fine.

---

Compute and compare the mean and std dev of train, validation and test datasets

In [ ]:
# Calculate mean and std on the provided dataset
def calculate_mean_std(dataset, batch_size=64):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize variables to store the sum of means and stds
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_batches = 0

    # Iterate through the DataLoader
    try:
        for batch in dataloader:
            images, _ = batch
            #print(images.size()) #torch.Size([64, 3, 224, 224])
            batch_samples = images.size(0)
            images = images.view(batch_samples, images.size(1), -1)
            #print(images.size()) #torch.Size([64, 3, 50176])
            mean += images.mean(2).sum(0)
            std += images.std(2).sum(0)
            total_batches += batch_samples
    except OSError:
        print(f"Error occurred while processing")

    # Calculate the mean and std
    mean /= total_batches
    std /= total_batches

    return mean, std

In [ ]:
mean, std = calculate_mean_std(train_dataset)
print(f"For train_dataset -- Mean: {mean}, Std: {std}")

mean, std = calculate_mean_std(valid_dataset)
print(f"For valid_dataset -- Mean: {mean}, Std: {std}")

mean, std = calculate_mean_std(test_dataset)
print(f"For test_dataset -- Mean: {mean}, Std: {std}")

As we can see above the mean and std dev over the RGB channels for all three datasets are similar. So thats good.